# Initialization and imports

In [1]:
#Celda exclusiva colab:
#"""
#Instalar dependencias
!pip install transformers adapters langid scikit-optimize mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.0/688.0 kB 54.8 MB/s eta 0:00:00
   

In [2]:
import os
os.chdir('/content')
!rm -rf VRID_language_proyect

!git clone --branch dev_Alonso_test --single-branch https://github.com/jitalo333/VRID_language_proyect
#Moverse a repositorio
os.chdir('/content/VRID_language_proyect/BERT')
#Montar drive
from google.colab import drive
drive.mount('/content/drive')
#"""

Cloning into 'VRID_language_proyect'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 169 (delta 71), reused 151 (delta 57), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 1.32 MiB | 6.91 MiB/s, done.
Resolving deltas: 100% (71/71), done.
Mounted at /content/drive


In [3]:
import pandas as pd
import os
from preprocess import clean_text
from encoder import embed_texts, prepare_data
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from transformers import MarianMTModel, MarianTokenizer
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM
from translate import translator, final_clean, gen_text_for_embedding
from translate import Helsinki_translate_esp_en, nllb_translate_esp_en

#Train model
import ast
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter
from ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, eval_model, safe_log_metric
from ML_pipeline_skp import mlflow_ckeckpoint

# 1) Preprocesamiento de los datos

In [4]:
# 1) Cargar datos
path = "/content/drive/MyDrive/VRID_NLP/code/VRID_proyect/data"
filePATH = os.path.join(path, "data_concatenada.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario", "Facultad del Proyecto",
                            "Depto Persona"]) \
       .fillna("")

# 2) Preprocesar los datos
#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}

df[list(cols.values())] = df[list(cols.keys())].map(clean_text)
df.head()
#Preprocesamiento de datos
savepath=os.path.join(path, "data/data_preprocessed.xlsx")
#df.to_excel(savepath, index=False)

In [5]:
df = df.iloc[0:10].copy()

# 2) Traducción del texto

## Translator

In [7]:
dfh = df.copy()
path = "/content/drive/MyDrive/VRID_NLP/code/VRID_proyect/"
#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer, Helsinki_translate_esp_en)

cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
dfh[list(cols.values())] = dfh[list(cols.keys())].map(trans.detect_and_translate)
#savepath=os.path.join(path, "data/data_translated.xlsx")
#df.to_excel(savepath, index=False)
dfh.to_excel('data_helsinki.xlsx', index=False)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

In [8]:
dfn = df.copy()

#1. Cargar modelo de traducción
path = "/content/drive/MyDrive/VRID_NLP/code/VRID_proyect/"
#model_name = "facebook/nllb-200-3.3B"  # O "facebook/nllb-200-distilled-600M" para algo más liviano
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = NllbTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
trans = translator(model, tokenizer, nllb_translate_esp_en)

cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}
#2. Traducción de columnas
dfn[list(cols.values())] = dfn[list(cols.keys())].map(trans.detect_and_translate)

#savepath=os.path.join(path, "data/data_translated.xlsx")
dfn.to_excel('data_nllb.xlsx', index=False)

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

## Last cleaning

In [ ]:
import re

def final_clean(text):
    if not isinstance(text, str):
        return ""
    # Reemplaza saltos de línea y tabs por un espacio
    text = re.sub(r'[\r\n\t]+', ' ', text)
    # Colapsa espacios múltiples
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def gen_text_for_embedding(df, cols):
    """
    df   : DataFrame de entrada
    cols : lista de nombres de columnas a procesar y concatenar
    """
    df = df.copy()
    # Aplica limpieza a cada columna especificada
    for col in cols:
        df[col] = df[col].apply(final_clean)
    # Concatena las columnas limpias en una nueva columna
    df["text_for_embedding_translated"] = df[cols].agg(" ".join, axis=1)
    return df

#path = "/content/drive/MyDrive/VRID_NLP/code/VRID_proyect/"
#filepath = os.path.join(path, "data/data_translated.xlsx")
#df = pd.read_excel(filepath)
cols = ["title", "keywords", "resume"]
df = gen_text_for_embedding(df, cols)

In [ ]:
savepath=os.path.join(path, "data/data_translated_concat.xlsx")
#df.to_excel(savepath, index=False)

# 3) Embedding text

In [ ]:
#path = "/content/drive/MyDrive/VRID_NLP/code/VRID_proyect/"
#filePATH = os.path.join(path, "data_concatenada_translated.xlsx")
#df = pd.read_excel(filePATH)
# 1) Preparar textos y etiquetas
texts, labels = prepare_data(df)

# 2) Calcular embeddings
# Parámetros modelo
BASE_MODEL = "allenai/specter2_base"
ADAPTER_NAME = "allenai/specter2"

emb_texts = embed_texts(texts, BASE_MODEL, ADAPTER_NAME)

# 3) Guardar embeddings y etiquetas
df_dataset = pd.DataFrame(columns=["Código VRID", "labels", "embedings"])
df_dataset["Código VRID"] = df.loc[labels.index, "Código VRID"]
df_dataset["labels"] = labels
df_dataset["embedings"] = emb_texts.tolist()
df_dataset.to_excel("dataset_embed_translated.xlsx", index=False)

tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

pytorch_adapter.bin:   0%|          | 0.00/3.59M [00:00<?, ?B/s]

Modelo SPECTER2 cargado correctamente.


# 4) Train classifier

In [ ]:
path = "/content/drive/MyDrive/VRID_NLP/code/VRID_proyect/"
filePATH = os.path.join(path, "dataset_embed_translated.xlsx")
df = pd.read_excel(filePATH)
df['embedings'] = df['embedings'].apply(lambda x: np.array(ast.literal_eval(x)))
df.head()


y = df['labels'].to_numpy()
X = df['embedings'].to_numpy()
X = np.vstack(X)

seed = 7
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=seed, stratify=y)

# 2. Elegir modelos a probar
model_keys = [
    #'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    #'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print(X_train.shape)
print(X_test.shape)
print("📊 Comienzo:", Counter(y_train))
print("📊 Comienzo:", Counter(y_test))

In [ ]:
# 4. Ejecutar entrenamiento, validación y test con tus funciones
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, n_iter=2, sample_weight_On = True)

print(results_val)
for model in models_dicc.values():
  results_test = eval_model(model, X_test, y_test)
  print(results_test)

experiment_name = 'test_functions'
mlflow_ckeckpoint(results_val, models_dicc, X_test, y_test, experiment_name)